# Build Walk & Bike Graphs — Luxembourg


## Folder structure

```
gravity/
├── build_graphs.ipynb          ← this notebook
├── data/
│   └── EU_DEM_Lux.tif          ← DEM raster (GeoTIFF)
└── cache/                      ← auto-created; stores GraphML
```

## Notes

- On first run the OSM network is downloaded from the Overpass API. If the server is busy, the download is retried automatically with increasing waits.
- `data/EU_DEM_Lux.tif` must be present before running.

## Outputs

- `G_walk` — undirected
- `G_bike` — directed

Raw graphs are cached to `cache/` as GraphML.


In [7]:
import os
import math
import time
import pandas as pd
import re
import numpy as np
import networkx as nx
import geopandas as gpd
import osmnx as ox
from pathlib import Path
from shapely.geometry import Point

In [2]:
# =========================
# SETTINGS
# =========================

# --- Project root ---
PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
CACHE_DIR    = PROJECT_ROOT / "cache"

# --- Define study area ---
USE_PLACE_QUERY = True
PLACE_NAME = {"country": "Luxembourg"}

# --- CRS ---
PROJECT_CRS = "EPSG:2169"

# --- DEM path (GeoTIFF) ---
DEM_PATH = str(DATA_DIR / "EU_DEM_Lux.tif")

# --- Walking impedance: Tobler + steps ---
STAIR_SPEED_M_PER_MIN = 42.0

# --- Bike impedance: e-bike rule ---
BASE_KMH             = 18.0
MIN_KMH              = 6.0
UPHILL_THRESHOLD_PCT = 3.0
PENALTY_PER_PCT      = 1.5
DOWNHILL_CAP_KMH     = 24.0

# --- Cache / output dirs ---
WALK_GRAPHML = str(CACHE_DIR / "lux_walk.graphml")
BIKE_GRAPHML = str(CACHE_DIR / "lux_bike.graphml")

# --- OSMnx settings ---
ox.settings.use_cache           = True
ox.settings.log_console         = True
ox.settings.requests_timeout    = 180
ox.settings.overpass_rate_limit = True

print("All good.")


All good.


In [3]:
# =========================
# HELPERS
# =========================

def ensure_dir(path) -> None:
    os.makedirs(path, exist_ok=True)


def tobler_speed_kmh(grade_dec: float) -> float:
    """Tobler hiking function: v = 6 * exp(-3.5 * |S + 0.05|)"""
    return 6.0 * math.exp(-3.5 * abs(grade_dec + 0.05))


def ebike_speed_kmh(grade_dec: float) -> float:
    """E-bike speed model with uphill penalty and capped downhill benefit."""
    g_pct = grade_dec * 100.0
    if g_pct >= 0:
        excess = max(0.0, g_pct - UPHILL_THRESHOLD_PCT)
        speed = BASE_KMH - PENALTY_PER_PCT * excess
        return max(MIN_KMH, speed)
    else:
        benefit = 1.0 + 0.02 * min(10.0, abs(g_pct))
        return min(DOWNHILL_CAP_KMH, BASE_KMH * benefit)


def drop_bad_edges_inplace(G: nx.MultiDiGraph, time_attr: str) -> None:
    """Remove edges where time_attr is missing / None / NaN — in-place."""
    bad = []
    for u, v, k, d in G.edges(keys=True, data=True):
        t = d.get(time_attr, None)
        if t is None or (isinstance(t, float) and math.isnan(t)):
            bad.append((u, v, k))
    G.remove_edges_from(bad)
    print(f"  Dropped {len(bad)} edges with missing '{time_attr}'")


def download_with_retries(fn, *args, max_tries=6, base_wait_s=30, **kwargs):
    """
    Call an OSMnx download function, retrying with exponential backoff when the
    Overpass server is busy (e.g. 502/504 'at capacity'). Alternates between
    Overpass endpoints on each retry. Already-downloaded responses are reused
    because ox.settings.use_cache = True. Does not change any network rules.
    """
    endpoints = [
        "https://overpass-api.de/api",
        "https://overpass.kumi.systems/api",
    ]
    # OSMnx 2.x uses 'overpass_url'; OSMnx 1.x uses 'overpass_endpoint'
    url_attr = "overpass_url" if hasattr(ox.settings, "overpass_url") else "overpass_endpoint"

    for attempt in range(1, max_tries + 1):
        setattr(ox.settings, url_attr, endpoints[(attempt - 1) % len(endpoints)])
        try:
            print(f"  Attempt {attempt}/{max_tries} via {getattr(ox.settings, url_attr)}")
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == max_tries:
                raise
            wait = base_wait_s * 2 ** (attempt - 1)
            print(f"  Download failed ({type(e).__name__}: {str(e)[:120]})")
            print(f"  Retrying in {wait}s...")
            time.sleep(wait)


print("All good.")


All good.


In [4]:
# =========================
# 1) LOAD / DOWNLOAD GRAPHS
# =========================

ensure_dir(CACHE_DIR)
ensure_dir(str(CACHE_DIR / "sensitivity_thresholds"))
ensure_dir(str(CACHE_DIR / "sensitivity_binary"))

if os.path.exists(WALK_GRAPHML) and os.path.exists(BIKE_GRAPHML):
    print("Loading cached graphs from GraphML...")
    G_walk = ox.load_graphml(WALK_GRAPHML)
    G_bike = ox.load_graphml(BIKE_GRAPHML)

else:
    print("Downloading OSM graph with OSMnx...")

    custom_filter = (
        '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|secondary|'
        'tertiary|unclassified|residential|living_street|service|cycleway|path|'
        'footway|pedestrian|steps"]'
        '["area"!~"yes"]'
    )

    if USE_PLACE_QUERY:
        G_all = download_with_retries(
            ox.graph_from_place,
            PLACE_NAME, custom_filter=custom_filter, simplify=True, retain_all=False
        )
    else:
        G_all = ox.graph_from_bbox(
            north, south, east, west, custom_filter=custom_filter,
            simplify=True, retain_all=False,
        )

    # ---- WALK GRAPH ----
    G_walk = G_all.copy()
    walk_remove = []
    WALK_EXCLUDE_HW = {"motorway", "motorway_link", "trunk", "trunk_link"}
    for u, v, k, d in G_walk.edges(keys=True, data=True):
        hw = d.get("highway", "")
        hw_set = set(hw) if isinstance(hw, list) else {hw}
        foot = d.get("foot", None)
        if hw_set & WALK_EXCLUDE_HW:
            walk_remove.append((u, v, k))
        elif foot == "no":
            walk_remove.append((u, v, k))
    G_walk.remove_edges_from(walk_remove)
    print(f"Walk graph: removed {len(walk_remove)} edges (motorway/trunk, foot=no)")

    # ---- BIKE GRAPH ----
    G_bike = G_all.copy()
    bike_remove = []
    BIKE_EXCLUDE_HW = {"motorway", "motorway_link", "trunk", "trunk_link"}
    for u, v, k, d in G_bike.edges(keys=True, data=True):
        hw = d.get("highway", "")
        hw_set = set(hw) if isinstance(hw, list) else {hw}
        bicycle = d.get("bicycle", None)
        is_steps = "steps" in hw_set
        if hw_set & BIKE_EXCLUDE_HW:
            bike_remove.append((u, v, k))
        elif is_steps:
            bike_remove.append((u, v, k))
        elif bicycle == "no":
            bike_remove.append((u, v, k))
    G_bike.remove_edges_from(bike_remove)
    print(f"Bike graph: removed {len(bike_remove)} edges (motorway/trunk, steps, bicycle=no)")

    # ---- ADD ELEVATION + GRADES (before projecting) ----
    print("Adding node elevations from DEM...")
    G_walk = ox.elevation.add_node_elevations_raster(G_walk, DEM_PATH, cpus=1)
    G_bike = ox.elevation.add_node_elevations_raster(G_bike, DEM_PATH, cpus=1)
    print("Computing edge grades...")
    G_walk = ox.elevation.add_edge_grades(G_walk)
    G_bike = ox.elevation.add_edge_grades(G_bike)
    print("Elevation and grades added.")

    # ---- PROJECT + SAVE  ----
    G_walk = ox.project_graph(G_walk, to_crs=PROJECT_CRS)
    G_bike = ox.project_graph(G_bike, to_crs=PROJECT_CRS)
    ensure_dir(CACHE_DIR)
    ox.save_graphml(G_walk, WALK_GRAPHML)
    ox.save_graphml(G_bike, BIKE_GRAPHML)
    print("Graphs saved to:", WALK_GRAPHML, "and", BIKE_GRAPHML)

# Ensure projected
if G_walk.graph.get("crs") != PROJECT_CRS:
    G_walk = ox.project_graph(G_walk, to_crs=PROJECT_CRS)
if G_bike.graph.get("crs") != PROJECT_CRS:
    G_bike = ox.project_graph(G_bike, to_crs=PROJECT_CRS)

# Convert walk to undirected
G_walk = ox.convert.to_undirected(G_walk)

# Keep only largest connected component
# (removes orphan nodes and tiny disconnected fragments from OSM data)
print("Cleaning disconnected components...")

walk_components = list(nx.connected_components(G_walk))
largest_walk = max(walk_components, key=len)
print(f"  Walk: {len(walk_components)} components, largest has {len(largest_walk)} nodes, "
      f"dropping {G_walk.number_of_nodes() - len(largest_walk)} orphan nodes")
G_walk = G_walk.subgraph(largest_walk).copy()

bike_components = list(nx.weakly_connected_components(G_bike))
largest_bike = max(bike_components, key=len)
print(f"  Bike: {len(bike_components)} components, largest has {len(largest_bike)} nodes, "
      f"dropping {G_bike.number_of_nodes() - len(largest_bike)} orphan nodes")
G_bike = G_bike.subgraph(largest_bike).copy()

print(f"\nWalk graph: {G_walk.number_of_nodes()} nodes, {G_walk.number_of_edges()} edges")
print(f"Bike graph: {G_bike.number_of_nodes()} nodes, {G_bike.number_of_edges()} edges")
print(f"Avg edges/node (walk): {G_walk.number_of_edges() / G_walk.number_of_nodes():.2f}")
print(f"Avg edges/node (bike): {G_bike.number_of_edges() / G_bike.number_of_nodes():.2f}")

  Attempt 1/6 via https://overpass-api.de/api
  Download failed (ResponseStatusCodeError: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. A)
  Retrying in 30s...
  Attempt 2/6 via https://overpass.kumi.systems/api
  Download failed (ReadTimeout: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=180))
  Retrying in 60s...
  Attempt 3/6 via https://overpass-api.de/api
Walk graph: removed 887 edges (motorway/trunk, foot=no)
Bike graph: removed 8893 edges (motorway/trunk, steps, bicycle=no)
Adding node elevations from DEM...
Computing edge grades...
Elevation and grades added.
Graphs saved to: /Users/fgsumer/Desktop/bike-sharing-PT-accessibility-equity/gravity/cache/lux_walk.graphml and /Users/fgsumer/Desktop/bike-sharing-PT-accessibility-equity/gravity/cache/lux_bike.graphml
Cleaning disconnected components...
  Walk: 443 components, largest has 99635 nodes, dropping 726 orphan 

In [5]:
import numpy as np

def check_elevation_grades(G, name):
    elevations = [d.get('elevation') for _, d in G.nodes(data=True)]
    missing_elev = sum(1 for e in elevations if e is None)
    valid_elev = [e for e in elevations if e is not None]
    
    grades = [d.get('grade') for _, _, d in G.edges(data=True)]
    missing_grade = sum(1 for g in grades if g is None)
    valid_grades = [g for g in grades if g is not None]
    
    print(f"\n{'='*40}")
    print(f"Graph: {name}")
    print(f"  Nodes: {G.number_of_nodes()} | Missing elevation: {missing_elev}")
    print(f"  Edges: {G.number_of_edges()} | Missing grade: {missing_grade}")
    if valid_elev:
        print(f"  Elevation range: {min(valid_elev):.1f}m – {max(valid_elev):.1f}m")
    if valid_grades:
        print(f"  Grade range: {min(valid_grades)*100:.1f}% – {max(valid_grades)*100:.1f}%")
        print(f"  Mean abs grade: {np.mean(np.abs(valid_grades))*100:.2f}%")

check_elevation_grades(G_walk, "Walk")
check_elevation_grades(G_bike, "Bike")


Graph: Walk
  Nodes: 99635 | Missing elevation: 0
  Edges: 135921 | Missing grade: 0
  Elevation range: 132.0m – 551.0m
  Grade range: -596.8% – 479.7%
  Mean abs grade: 4.86%

Graph: Bike
  Nodes: 97218 | Missing elevation: 0
  Edges: 246310 | Missing grade: 0
  Elevation range: 132.0m – 551.0m
  Grade range: -535.1% – 535.1%
  Mean abs grade: 4.72%


In [6]:
# =========================
# 3) COMPUTE EDGE TRAVEL TIMES
# =========================

# Clip grades to physically plausible and
# artifacts on edges shorter than the DEM cell size (25 m).
GRADE_CAP = 0.40   # ±40% covers the steepest real pedestrian/e-bike infrastructure


def safe_grade(raw) -> float:
    """Return a finite grade clipped to ±GRADE_CAP; treat None/NaN as 0.0."""
    if raw is None:
        return 0.0
    try:
        g = float(raw)
    except (TypeError, ValueError):
        return 0.0
    if math.isnan(g):
        return 0.0
    return max(-GRADE_CAP, min(GRADE_CAP, g))


print("Computing walk_time_min on edges...")
for u, v, k, d in G_walk.edges(keys=True, data=True):
    L = float(d.get("length", 0.0))
    grade = safe_grade(d.get("grade"))
    hw = d.get("highway", None)
    is_steps = ("steps" in hw) if isinstance(hw, list) else (hw == "steps")

    if L <= 0:
        d["walk_time_min"] = None
        continue
    if is_steps:
        d["walk_time_min"] = L / STAIR_SPEED_M_PER_MIN
    else:
        speed_kmh = tobler_speed_kmh(grade)
        d["walk_time_min"] = ((L / 1000.0) / speed_kmh) * 60.0

print("Computing bike_time_min on edges...")
for u, v, k, d in G_bike.edges(keys=True, data=True):
    L = float(d.get("length", 0.0))
    grade = safe_grade(d.get("grade"))

    if L <= 0:
        d["bike_time_min"] = None
        continue
    speed_kmh = ebike_speed_kmh(grade)
    d["bike_time_min"] = ((L / 1000.0) / speed_kmh) * 60.0

drop_bad_edges_inplace(G_walk, "walk_time_min")
drop_bad_edges_inplace(G_bike, "bike_time_min")

print("All good with travel times.")

Computing walk_time_min on edges...
Computing bike_time_min on edges...
  Dropped 0 edges with missing 'walk_time_min'
  Dropped 0 edges with missing 'bike_time_min'
All good with travel times.
